# Phase 10: Needleman-Wunsch CUDA Implementation


This notebook validates the Phase 10 CUDA prototype for Needleman-Wunsch global alignment. The GPU implementation uses wavefront parallelism across DP anti-diagonals and validates scores against the CPU Needleman-Wunsch reference before future Smith-Waterman CUDA work.

In [ ]:
!nvidia-smi
!nvcc --version


In [ ]:
import os
print("Current working directory:", os.getcwd())


Compile the CPU reference and CUDA Needleman-Wunsch executable.

In [ ]:
!g++ src/needleman_wunsch_cpu.cpp \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o needleman_wunsch_cpu

!nvcc src/needleman_wunsch_gpu.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o needleman_wunsch_gpu


Compile and run the GPU validation test runner.

In [ ]:
!g++ tests/test_needleman_wunsch_gpu.cpp \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o test_needleman_wunsch_gpu


In [ ]:
!./test_needleman_wunsch_gpu


Generate a small fixed-length synthetic dataset for a direct GPU run.

In [ ]:
!python scripts/generate_synthetic_dataset.py \
  --num-pairs 100 \
  --sequence-length 32 \
  --output data/synthetic/synthetic_pairs_32.txt \
  --seed 42


In [ ]:
!mkdir -p results/needleman_wunsch

!./needleman_wunsch_gpu \
  data/synthetic/synthetic_pairs_32.txt \
  results/needleman_wunsch/needleman_wunsch_gpu_results.csv \
  --repetitions 5 \
  --implementation wavefront


Run the CPU vs GPU benchmark and generate charts.

In [ ]:
!python benchmarks/run_needleman_wunsch_gpu_benchmark.py


In [ ]:
!python scripts/plot_needleman_wunsch_gpu_benchmark.py


Display benchmark results.

In [ ]:
import pandas as pd
df = pd.read_csv("benchmarks/needleman_wunsch_gpu_benchmark_results.csv")
df


Display generated charts.

In [ ]:
from pathlib import Path
from IPython.display import display, Image

chart_directory = Path("assets/benchmark_charts/needleman_wunsch_gpu")
for chart_path in sorted(chart_directory.glob("*.png")):
    print(chart_path)
    display(Image(filename=str(chart_path)))
